# Day 29 · 量化与推理加速

**配套讲义**: [`days/day-29.md`](../days/day-29.md) ｜ **需要 GPU（云机器）**

对 SFT/DPO 后的模型做 AWQ 量化，量出**显存 / 首 token 延迟 / 吞吐**三项指标的前后对比；并解释为什么量化对 VLM 的**视觉塔**尤其敏感。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w5.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys, torch
print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cuda   :", torch.version.cuda, "| available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"gpu    : {p.name}  {p.total_memory / 1024**3:.0f} GB")
    print("bf16   :", torch.cuda.is_bf16_supported())
else:
    print("⚠️  没有 GPU —— 这一天的训练/推理跑不了。先看 docs/13-hardware-and-cost.md 租机器")

## 1. 先算账

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "src.serve.quantize",
                    "--model", "outputs/qwen25vl3b-cx-dpo-v0", "--estimate"],
                   capture_output=True, text=True, cwd="..")
print(r.stdout or r.stderr)

## 2. 看排除列表 —— 今天最该记住的 6 行代码

In [ ]:
import sys; sys.path.insert(0, "..")
from src.serve.quantize import VISION_EXCLUDE, build_quant_config, print_quality_warning

print("不参与量化的模块前缀:")
for name in VISION_EXCLUDE:
    print("   ", name)
print("\n量化配置:")
try:
    cfg = build_quant_config(exclude=VISION_EXCLUDE)
    print(cfg)
except Exception as e:
    print("（需要 GPU 环境）", e)
print()
print_quality_warning()

## 3. 反例实验设计

设计一个能**证明视觉塔敏感**的小实验：

1. 准备 20 张带文字的图 + 对应 OCR 问题
2. 分别用「混合精度」和「全量化」两个模型跑
3. 对比准确率

**这个实验做出来，你就真的懂了为什么 VLM 量化要区别对待。**

In [ ]:
experiment_plan = """
对照组：awq 混合精度（视觉塔 bf16）
实验组：awq 全量化
样本：20 张带文字的图
指标：OCR 准确率
预期：实验组下降 ___ 个百分点
"""
print(experiment_plan)

## 验收清单

- [ ] 三项指标（显存 / 首 token 延迟 / 吞吐）的前后对比表已产出
- [ ] 能解释为什么视觉塔要保持高精度（激活值分布 + 细粒度信息）
- [ ] calibration 数据里**必须含图** —— 且你能说出为什么
- [ ] vLLM 服务能用 `curl` 打通，知道 `--limit-mm-per-prompt` 是干嘛的

**卡住了？** 回看 [`days/day-29.md`](../days/day-29.md) 第五节「容易踩的坑」。

> **明天**：`days/day-30.md` —— 端到端推理服务，W5 收官